# Name That LLM · V2 风格特征融合

使用官方 `train.csv` 训练词级 + 字符级 TF-IDF 与风格统计特征的线性 SVM；Notebook 输出 `/kaggle/working/submission.parquet`。本地五折 Prompt-disjoint OOF BA = **0.358553**，并非 Kaggle 公榜结果。


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

def find_competition_file(name):
    for folder in [
        Path('/kaggle/input/competitions/name-that-llm'),
        Path('/kaggle/input/name-that-llm')
    ]:
        p = folder/name
        if p.is_file():
            return p
    raise FileNotFoundError('Competition dataset missing: ' + name)

train = pd.read_csv(find_competition_file('train.csv'))
test = pd.read_csv(find_competition_file('test.csv'))
assert list(train.columns) == ['id','prompt_id','text','model']
assert list(test.columns) == ['id','text']
assert train.model.nunique() == 18
assert train[['id','text','model']].notna().all().all()
assert test[['id','text']].notna().all().all()
assert test.id.is_unique
print('train',train.shape,'test',test.shape)


In [ ]:
TOKENS = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?", re.UNICODE)
SENTENCES = re.compile(r"[.!?]+(?:\s|$)")
WORD_SET = [
    "the", "and", "of", "to", "in", "is", "that", "this", "which",
    "with", "as", "by", "for", "on", "from", "are", "can", "may", "might",
    "will", "should", "could", "we", "you", "i", "they", "it", "not",
    "but", "or", "a", "an", "their", "its", "our", "your",
    "however", "therefore", "indeed", "perhaps", "ultimately",
    "although", "while", "because", "rather", "instead", "through",
    "each", "every", "even", "yet", "only", "often", "also",
]
PHRASES = [
    "for example", "such as", "in fact", "rather than",
    "not only", "not just", "on the other hand", "in other words",
    "at the same time", "in this way", "as a result",
]
PUNCT = list(",;:.!?-—–()[]{}\"'“”‘’/\\*#\n")


def style_vector(text: str) -> np.ndarray:
    """Surface-only, topic-agnostic style statistics."""
    words = [w.lower() for w in TOKENS.findall(text)]
    n = max(1, len(words))
    chars = max(1, len(text))
    lower = text.lower()
    count = {}
    for w in words:
        count[w] = count.get(w, 0) + 1
    counts = np.asarray(list(count.values()), dtype=float) if count else np.array([0.0])
    sent_len = [len(TOKENS.findall(s)) for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if not sent_len:
        sent_len = [n]
    words_len = [len(w) for w in words] or [0]
    vals = [
        np.log1p(chars), np.log1p(n),
        float(np.mean(words_len)), float(np.std(words_len)),
        np.mean(sent_len), np.std(sent_len),
        len(count) / n, np.sum(counts == 1) / n,
        np.sum(counts >= 3) / n,
        sum(len(w) >= 8 for w in words) / n,
        sum(len(w) <= 3 for w in words) / n,
        sum(w[0].isupper() for w in TOKENS.findall(text)) / n,
        sum(c.isupper() for c in text) * 100 / chars,
        sum(c.isdigit() for c in text) * 100 / chars,
        sum(ord(c) > 127 for c in text) * 100 / chars,
        sum(c.isspace() for c in text) * 100 / chars,
        text.count("\n\n") * 100 / chars,
        len(re.findall(r"\b\d+(?:\.\d+)?\b", text)) / n,
    ]
    vals.extend([text.count(p) * 100 / chars for p in PUNCT])
    vals.extend([count.get(w, 0) * 100 / n for w in WORD_SET])
    vals.extend([lower.count(p) * 100 / n for p in PHRASES])
    vals.append(sum(text.startswith(prefix) for prefix in (
        "In ", "The ", "A ", "When ", "Imagine ", "Consider ", "There ", "Every ",
    )))
    return np.asarray(vals, dtype=np.float32)


def style_matrix(texts) -> np.ndarray:
    return np.asarray([style_vector(text) for text in texts], dtype=np.float32)

In [ ]:
char = TfidfVectorizer(
    analyzer="char", ngram_range=(3,5), min_df=2, max_features=100000,
    sublinear_tf=True, dtype=np.float32
)
word = TfidfVectorizer(
    analyzer="word", ngram_range=(1,2), min_df=2, max_features=80000,
    sublinear_tf=True, dtype=np.float32
)
xtrain = train.text.tolist()
xtest = test.text.tolist()
chr_tr, chr_te = char.fit_transform(xtrain), char.transform(xtest)
word_tr, word_te = word.fit_transform(xtrain), word.transform(xtest)
scaler = StandardScaler()
style_tr = scaler.fit_transform(style_matrix(xtrain))
style_te = scaler.transform(style_matrix(xtest))
style_tr = csr_matrix(np.clip(style_tr, -4, 4))
style_te = csr_matrix(np.clip(style_te, -4, 4))
features_tr = hstack([chr_tr, word_tr, style_tr * 0.08], format="csr")
features_te = hstack([chr_te, word_te, style_te * 0.08], format="csr")
clf = LinearSVC(C=1.0, max_iter=4000)
clf.fit(features_tr, train.model)
pred = clf.predict(features_te)

submission = pd.DataFrame({'id': test.id, 'model': pred})
assert submission.id.equals(test.id) and submission.id.is_unique
assert submission.model.isin(train.model.unique()).all()
assert submission.notna().all().all()
output = Path('/kaggle/working/submission.parquet') if Path('/kaggle/working').is_dir() else Path('submission.parquet')
submission.to_parquet(output, index=False)
print('saved',output,'rows:',len(submission))
print(submission.head())
